## Tool de Wikipedia

`wikipedia_search(query, num_results=3)` cerca pàgines amb la Search API i recupera el wikitext de cadascuna amb la Get Page API. Retorna una llista de diccionaris amb `title`, `url` i `content`, preparada per serialitzar com a JSON quan connectem l'agent.

Fem servir `params` perquè `requests` codifiqui els espais i els caràcters especials. Totes les peticions inclouen el `User-Agent` requerit i un timeout de 20 segons. Els errors HTTP o de xarxa es propaguen al codi que cridi la tool; una cerca sense coincidències retorna `[]`.


In [33]:
import requests
from urllib.parse import quote


WIKIPEDIA_HEADERS = {
    "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
}


def search_wikipedia(query: str, num_results: int | None = None) -> list[dict]:
    """Find English Wikipedia pages related to a non-empty query."""
    if not query.strip():
        raise ValueError("query must not be empty")

    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
    }
    if num_results is not None:
        params["srlimit"] = num_results

    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params=params,
        headers=WIKIPEDIA_HEADERS,
        timeout=20,
    )
    response.raise_for_status()
    data = response.json()
    if "error" in data:
        raise ValueError(f"Wikipedia API error: {data['error']}")
    return data["query"]["search"]


def get_page(title: str) -> str:
    """Fetch raw wikitext using the exact title returned by search."""
    response = requests.get(
        "https://en.wikipedia.org/w/index.php",
        params={"title": title, "action": "raw"},
        headers=WIKIPEDIA_HEADERS,
        timeout=20,
    )
    response.raise_for_status()
    return response.text


def wikipedia_search(query: str, num_results: int = 3) -> list[dict]:
    """Search English Wikipedia and return titles, URLs and raw page content."""
    pages = search_wikipedia(query, num_results)
    return [
        {
            "title": page["title"],
            "url": "https://en.wikipedia.org/wiki/"
            + quote(page["title"].replace(" ", "_"), safe=""),
            "content": get_page(page["title"]),
        }
        for page in pages
    ]


In [34]:
results = search_wikipedia("capybara")
print(f"Question1: {len(results)}")


print(f"Question2: {sum('capybara' in r['title'].lower() for r in results)}")

page_content = get_page("Capybara")
print(f"Question3: {len(page_content)}")

Question1: 10
Question2: 6
Question3: 44767


In [46]:
from pydantic_ai import Agent

agent = Agent(
    "openai:gpt-4o-mini",
    instructions=(
        "Respon les preguntes consultant Wikipedia. "
        "Fes servir search_wikipedia per trobar pàgines rellevants "
        "i get_page per llegir-ne el contingut. "
        "Inclou els títols de les pàgines consultades a la resposta."
    ),
    tools=[search_wikipedia, get_page],
)

framework = agent.__class__.__module__.split(".")[0]
provider = agent.model.system

print(f"Question4: Framework: {framework}. LLM provider: {provider}.")

Question4: Framework: pydantic_ai. LLM provider: openai.


In [49]:
print("Question5:")
result = await agent.run(
    "What is this page about? https://en.wikipedia.org/wiki/Capybara"
)

from pydantic_ai.messages import ToolCallPart, ToolReturnPart

for message in result.new_messages():
    for part in message.parts:
        if isinstance(part, ToolCallPart):
            print(f"Tool sol·licitada: {part.tool_name}")
            print(f"Arguments: {part.args_as_dict()}")

        elif isinstance(part, ToolReturnPart):
            print(f"Resultat de: {part.tool_name}")
            print(f"Contingut retornat: {str(part.content)[:200]}...\n")

print("Resposta final:")
print(result.output)

Question5:
Tool sol·licitada: search_wikipedia
Arguments: {'query': 'Capybara'}
Resultat de: search_wikipedia
Contingut retornat: [{'ns': 0, 'title': 'Capybara', 'pageid': 6776, 'size': 44903, 'wordcount': 4312, 'snippet': 'The <span class="searchmatch">capybara</span> (/kæpɪˈbɑːrə/ kap-ih-BAR-uh) or greater <span class="searchm...

Tool sol·licitada: get_page
Arguments: {'title': 'Capybara'}
Resultat de: get_page
Contingut retornat: {{Short description|Largest species of rodents}}
{{Other uses}}
{{Good article}}
{{pp|small=yes}}
{{Use dmy dates|date=July 2022}}
{{Speciesbox
| status = LC
| status_system = IUCN3.1
| status_ref = <...

Resposta final:
The Wikipedia page on the **Capybara** details this animal as the largest living rodent, specifically classified as *Hydrochoerus hydrochaeris*. It is primarily native to South America, excluding Chile. The capybara is a semi-aquatic herbivore typically found in savannas and forests near freshwater sources.

### Key Points from the Article

In [55]:
print("Question6:")
result = await agent.run(
    "What are the main threats to capybara populations?"
)
# from pydantic_ai.messages import ToolCallPart, ToolReturnPart

# for message in result.new_messages():
#     for part in message.parts:
#         if isinstance(part, ToolCallPart):
#             print(f"Tool sol·licitada: {part.tool_name}")
#             print(f"Arguments: {part.args_as_dict()}")

#         elif isinstance(part, ToolReturnPart):
#             print(f"Resultat de: {part.tool_name}")
#             print(f"Contingut retornat: {str(part.content)[:200]}...\n")

# print("Resposta final:")
# print(result.output)

question6_tool_calls = [
    part
    for message in result.new_messages()
    for part in message.parts
    if isinstance(part, ToolCallPart)
]

print(f"Total tool calls: {len(question6_tool_calls)}")
for number, call in enumerate(question6_tool_calls, start=1):
    print(f"{number}. {call.tool_name}: {call.args_as_dict()}")

print("\nFinal answer:")
print(result.output)

Question6:
Total tool calls: 3
1. search_wikipedia: {'query': 'Capybara threats'}
2. search_wikipedia: {'query': 'Capybara'}
3. get_page: {'title': 'Capybara'}

Final answer:
The main threats to capybara populations include:

1. **Hunting**: Capybaras are hunted for their meat and skin. In some areas, they are farmed, which while helping preserve their habitat, can also contribute to population declines if not managed sustainably.

2. **Habitat Loss**: As their natural wetlands are drained for agriculture or urbanization, capybaras lose their habitat and sources of food. This is particularly concerning in regions experiencing rapid development.

3. **Human-Wildlife Conflict**: In some cases, capybaras are viewed as pests by farmers as they graze on grasses that are also valuable livestock feed. This can lead to capybaras being killed in retaliation.

4. **Predation**: In the wild, capybaras face threats from natural predators such as jaguars, ocelots, caimans, and birds of prey. Althou